In [ ]:
import os
import pandas as pd
import re

# ==========================================
# 0. 구글 드라이브에서 파일 자동 다운로드
# ==========================================
pip install gdown
import gdown

file_name = 'reddit_opinion_PSE_ISR.csv'
file_id = '1BV33DIzgZV9Ypb-tY6N7UxGhdR1FCzsI' 

if not os.path.exists(file_name):
    print(f"⏳ '{file_name}' 파일이 없습니다. 구글 드라이브에서 다운로드를 시작합니다...")
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, file_name, quiet=False)
    print("✅ 파일 다운로드 완료!\n")
else:
    print(f"✅ '{file_name}' 파일이 이미 존재합니다. 다운로드를 건너뜁니다.\n")

# ==========================================
# 1. 데이터 로드 및 기본 결측치 제거
# ==========================================
df = pd.read_csv(file_name)

# 결측치 및 삭제된 글 제거
df = df.dropna(subset=['self_text'])
df = df[~df['self_text'].astype(str).isin(['[deleted]', '[removed]', 'nan', ''])]

# ==========================================
# 2. 날짜 필터링 (9월 7일 ~ 11월 7일)
# ==========================================
# 타임존 이슈를 방지하기 위해 utc=True 후 타임존 제거
df['created_time'] = pd.to_datetime(df['created_time'], utc=True).dt.tz_localize(None)

start_date = pd.Timestamp('2023-09-07')
end_date = pd.Timestamp('2023-11-07 23:59:59')

# 해당 기간만 필터링 (전쟁 전/후 1달)
df = df[(df['created_time'] >= start_date) & (df['created_time'] <= end_date)]

# ==========================================
# 3. 감정 보존형 텍스트 전처리 함수
# ==========================================
def preprocess_for_emotion(text):
    text = str(text)
    
    # 1) 불필요한 링크 및 태그 제거 (노이즈만 제거)
    text = re.sub(r'http\S+|www\S+', '', text)  # URL 제거
    text = re.sub(r'&amp;|&lt;|&gt;', '', text) # HTML 엔티티 제거
    
    # 2) 텍스트 정규화 (의미 없는 반복 문자나 여백 정리)
    text = re.sub(r'\s+', ' ', text).strip()    # 다중 공백 및 줄바꿈을 띄어쓰기 하나로 압축
    
    # [중요] 불용어(Stopwords) 제거 안 함! -> 문맥 보존 (not, never 등 살림)
    # [중요] 구두점(!, ?, .) 제거 안 함! -> 감정 강도 보존
    # [중요] 소문자화(lower) 안 함! -> 대문자 강조(ex. REALLY) 보존
    
    return text

# ==========================================
# 4. 전처리 적용 및 빈 데이터 2차 정리
# ==========================================
df['cleaned_text'] = df['self_text'].apply(preprocess_for_emotion)

# URL 등만 있어서 전처리 후 빈칸이 되어버린 데이터 삭제
df = df[df['cleaned_text'] != '']

# ==========================================
# 5. 최종 데이터 저장
# ==========================================
df.to_csv('modified_preprocessing.csv', index=False, encoding='utf-8')

print("="*60)
print(f"✅ 전처리 완료! 총 {len(df)}개의 데이터가 모든 컬럼과 함께 저장되었습니다.")
print(f"📁 저장 파일명: modified_preprocessing.csv")
print(f"📊 저장된 컬럼 수: {len(df.columns)}개")
print(f"📅 최종 데이터 기간: {df['created_time'].min().date()} ~ {df['created_time'].max().date()}")
print("="*60)